In [ ]:
# Setup

# pip.main(['install', 'splink'])
# pip.main(['install', 'pyspark'])
# pip.main(['install', 'duckdb'])
# pip.main(['install', 'pyarrow'])
# pip.main(['install', 'pandas'])

In [73]:
# package imports

from itertools import count
import pip
from requests import head
import splink
import pyspark 
import pandas as pd
import re 
import pyarrow as pa


# SPINK setup
import splink.comparison_library as cl
import splink.comparison_level_library as cll
from splink.exploratory import profile_columns
from splink.comparison_library import CustomComparison
import duckdb, os, tempfile
import sys
from splink.blocking_analysis import (
    cumulative_comparisons_to_be_scored_from_blocking_rules_chart,
)
from splink import DuckDBAPI, Linker, SettingsCreator, block_on
from splink.exploratory import completeness_chart
import csv



In [74]:
# functions


def cleanse_names(series: pd.Series) -> pd.Series:
    """
    Clean text columns similar to your Spark UDF logic.
    """
    # lowercase
    cleaned = series.str.lower()

    # remove special characters (keep only letters, digits, space)
    cleaned = cleaned.str.replace(r"[^a-z0-9 ]", "", regex=True)

    # normalize whitespace
    cleaned = cleaned.str.strip().str.replace(r"\s+", " ", regex=True)

    # replace empty strings with None/NaN
    cleaned = cleaned.replace("", pd.NA)

    return cleaned


In [75]:
# DuckDB setup

# Set up DuckDB in memory
# In theory we can set this to a path on the local drive, but it will be slower
con = duckdb.connect(":memory:")

# Set up temporary dir for disk spilling.
spill_dir = tempfile.mkdtemp(prefix="duckdb_spill_")
con.execute("SET memory_limit = '100GB';")  # synonyms: max_memory / memory_limit
con.execute(f"SET temp_directory = '{spill_dir}';")
con.execute("SET max_temp_directory_size = '200GB';")

# This gets used across various splink functions
db_api = DuckDBAPI(con)

In [77]:

# read gias data
gias = pd.read_csv('Data/gias_data2024-03-01_2024-09-01_28.csv')

C:\Users\spilling\AppData\Local\Temp\ipykernel_14232\2644862179.py:2: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  gias = pd.read_csv('Data/gias_data2024-03-01_2024-09-01_28.csv')


In [78]:

# clean nulls 

for col_name in gias.columns:
    gias[col_name] = gias[col_name].replace(
        to_replace=[r"^\s*$", r"^NA$", r"^NA NA$", r"^na$", r"^NaN$", r"^nan$", r"^N/A$", r"^n/a$"],
        value=pd.NA,
        regex=True
    )

# issue with mixed data types in laestab
gias['laestab'] = gias['laestab'].astype(str)
gias['easting'] = gias['easting'].astype(float)
gias['northing'] = gias['northing'].astype(float)

cols_to_clean = [
    "heads_name",
    "establishment_name",
    "previous_establishment_number",
    "trusts_name",
]

for c in cols_to_clean:
    gias[c] = cleanse_names(gias[c].astype(str))


    # Ensure all columns are strings for Splink
    for col in gias.columns:
        gias[col] = gias[col].astype(str)

# drop unnamed index column if it exists

gias = gias.drop(columns=['Unnamed: 0'])

# Drop duplicates ignoring 'gias_date' column

subset = gias.columns.difference(['gias_date'])

gias = gias.drop_duplicates(subset=subset)


gias["unique_id"] = range(1, len(gias) + 1)



In [79]:
print(gias.columns)

print(len(gias))

Index(['urn', 'establishment_name', 'previous_la_code',
       'previous_establishment_number', 'northing', 'easting', 'postcode',
       'ukprn', 'establishment_type_group_name', 'phase_of_education_name',
       'establishment_status_name', 'trusts_name', 'official_sixth_form_name',
       'administrative_ward_name', 'msoa_name', 'lsoa_name', 'uprn',
       'school_capacity', 'gender_name', 'statutory_low_age',
       'statutory_high_age', 'open_date', 'close_date', 'gias_date', 'laestab',
       'previous_laestab', 'heads_name', 'unique_id'],
      dtype='object')
56172


In [80]:

completeness_chart(
    gias,
    db_api=db_api)

alt.LayerChart(...)

In [81]:
profile_columns(gias, db_api=db_api, column_expressions=["easting"])


alt.VConcatChart(...)

In [ ]:
profile_columns(gias, db_api=db_api, column_expressions=["heads_name"])


In [ ]:
blocking_rules_link = [
  block_on("urn"),
  block_on("establishment_name"),
  block_on("laestab"),
  # block_on("ukprn"),
  # block_on("northing", "easting"),
   block_on("postcode"),
  # block_on("heads_name"),
  # block_on("phase_of_education_name", "postcode"),
]

cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
  table_or_tables=gias,
  blocking_rules=blocking_rules_link,
  db_api=db_api,
  link_type="dedupe_only",
)


In [ ]:
# custom comparison for full_name

headteacher_name_comparison = CustomComparison(
    output_column_name="heads_name",
    comparison_levels=[
        cll.NullLevel("heads_name"),
        cll.ExactMatchLevel("heads_name").configure(tf_adjustment_column="heads_name"),
        cll.JaroWinklerLevel("heads_name", 0.9).configure(tf_adjustment_column="heads_name"),
        cll.ElseLevel(),
    ],
)

northing_easting_comparison = CustomComparison(
     output_column_name="northing_easting_distance",
     comparison_levels=[
         cll.And(cll.NullLevel("easting"),cll.NullLevel("northing")),  # level 0: nulls
         cll.And(cll.ExactMatchLevel("easting"),cll.ExactMatchLevel("northing")),
         cll.CustomLevel(
             """ (easting_l IS NOT NULL AND northing_l IS NOT NULL AND
                  easting_r IS NOT NULL AND northing_r IS NOT NULL AND
                  SQRT((CAST(easting_l as INT) - CAST(easting_r as INT))*(CAST(easting_l as INT) - CAST(easting_r as INT)) +
                   (CAST(northing_l as INT) - CAST(northing_r as INT))*(CAST(northing_l as INT) - CAST(northing_r as INT))) < 350)
             """
         ),
         cll.ElseLevel()  # everything else
     ]
 )

In [ ]:


settings = SettingsCreator(
    link_type="dedupe_only",
    unique_id_column_name="unique_id",
    # probability_two_random_records_match=1e-6,  # very small
    blocking_rules_to_generate_predictions=blocking_rules_link,
    comparisons=[
        cl.ExactMatch("urn"),
        cl.ExactMatch("laestab"),
        cl.PostcodeComparison("postcode",km_thresholds=[1, 10, 100]),
        cl.NameComparison("establishment_name"),
        headteacher_name_comparison,
        northing_easting_comparison
    ],
    retain_intermediate_calculation_columns=True,
)

linker = Linker(
    gias,
    settings,
    db_api=db_api,
    validate_settings=True,
)

In [ ]:

linker.training.estimate_probability_two_random_records_match(
    [
        #  block_on("establishment_name"),
        block_on("urn"),
       # block_on("laestab"),
        #block_on("postcode"),
       # block_on("heads_name"),
      #  block_on("trusts_name"),
     #   block_on("northing", "easting"),
    ],
    recall=0.95,
)


In [ ]:
linker.training.estimate_u_using_random_sampling(max_pairs=1e7)


training_blocking_rule = block_on("urn")

training_session_names = (
    linker.training.estimate_parameters_using_expectation_maximisation(
        training_blocking_rule, estimate_without_term_frequencies=True
    )
)


In [ ]:

linker.training.estimate_parameters_using_expectation_maximisation(
    blocking_rule=block_on("establishment_name"),
)

linker.training.estimate_parameters_using_expectation_maximisation(
    blocking_rule=block_on("laestab"),
)


In [ ]:
linker.visualisations.parameter_estimate_comparisons_chart()



In [ ]:

linker.visualisations.match_weights_chart()

In [ ]:

linker.evaluation.unlinkables_chart()

In [ ]:
df_predict = linker.inference.predict()

df_e = df_predict.as_pandas_dataframe()

df_e = df_e.sort_values(by="match_probability", ascending=False)


filtered_df = df_e[df_e["match_probability"] > 0.6]

print(f"Number of rows in that match: {len(filtered_df)}")

print(f"Match percentage: {len(filtered_df)/len(df_e)}")

In [ ]:
edge_records = df_e[(df_e["match_probability"] > threshold - 0.05) & (df_e["match_probability"] < threshold + 0.05)]


In [ ]:
threshold = 0.6
edge_records = df_e[(df_e["match_probability"] > threshold - 0.05) & (df_e["match_probability"] < threshold + 0.05)]

display(edge_records)

In [ ]:
records_to_plot = edge_records.head(200).to_dict(orient="records")


linker.visualisations.waterfall_chart(records_to_plot, filter_nulls=False)

In [ ]:
threshold = 0.6
edge_records = df_e[(df_e["match_probability"] > threshold - 0.05) & (df_e["match_probability"] < threshold + 0.05)]
display(edge_records)




In [ ]:
pip.main(['install', 'matplotlib'])

In [ ]:
df_e[(df_e['match_probability'] > 0.5) & (df_e['match_probability'] < 0.95)]['match_probability']

In [ ]:
import matplotlib.pyplot as plt

plt.hist(
    df_e[(df_e['match_probability'] > 0.5) & (df_e['match_probability'] < 0.95)]['match_probability'],
    bins=50,
    edgecolor='black'
)
plt.xlabel('Match Probability')
plt.ylabel('Frequency')
plt.title('Histogram of Match Probability between 0.6 and 0.95')
plt.show()